Save and export the best model using joblib or pickle while ensuring model versioning, storage efficiency, and compatibility for deployment without using default serialization functions.

In [4]:
import pandas as pd
import numpy as np
import joblib
import sklearn
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

# Load dataset
df = pd.read_csv("heart_disease_train.csv")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
target = "target"

df = df[features + [target]].copy()

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=int)

# Split dataset
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Define models
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42
    ),
    "SVM": make_pipeline(
        StandardScaler(),
        SVC(probability=True, class_weight="balanced", random_state=42)
    ),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    )
}

# Train models
for name, model in models.items():
    model.fit(X_train, y_train)

# Calculate metrics manually
def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return precision, recall, f1

# Select the best model using validation data
results = []

for name, model in models.items():
    pred = model.predict(X_val)
    precision, recall, f1 = calculate_metrics(y_val, pred)

    score = 0.20 * precision + 0.40 * recall + 0.40 * f1

    results.append({
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Custom Score": score
    })

results_df = pd.DataFrame(results).sort_values(
    "Custom Score", ascending=False
)

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Model comparison:")
print(results_df.round(4).to_string(index=False))
print("\nSelected model:", best_model_name)

# Save versioned model
model_dir = Path("saved_models")
model_dir.mkdir(exist_ok=True)

model_version = "v1"
model_path = model_dir / f"heart_disease_model_{model_version}.joblib"
metadata_path = model_dir / f"heart_disease_model_{model_version}_metadata.json"

joblib.dump(best_model, model_path, compress=3, protocol=4)

# Save metadata
metadata = {
    "model_name": best_model_name,
    "model_version": model_version,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "features": features,
    "validation_metrics": {
        "precision": float(results_df.iloc[0]["Precision"]),
        "recall": float(results_df.iloc[0]["Recall"]),
        "f1_score": float(results_df.iloc[0]["F1 Score"]),
        "custom_score": float(results_df.iloc[0]["Custom Score"])
    },
    "scikit_learn_version": sklearn.__version__,
    "joblib_version": joblib.__version__
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print("\nModel saved to:", model_path)
print("Metadata saved to:", metadata_path)
print("File size:", round(model_path.stat().st_size / 1024, 2), "KB")

# Verify saved model
loaded_model = joblib.load(model_path)

if np.array_equal(best_model.predict(X_test), loaded_model.predict(X_test)):
    print("Model verification successful.")
else:
    print("Warning: predictions differ.")

Model comparison:
              Model  Precision  Recall  F1 Score  Custom Score
Logistic Regression     0.9167  0.9167    0.9167        0.9167
                KNN     0.7500  1.0000    0.8571        0.8929
                SVM     0.9091  0.8333    0.8696        0.8630
      Random Forest     0.8333  0.8333    0.8333        0.8333
      Decision Tree     0.8333  0.8333    0.8333        0.8333

Selected model: Logistic Regression

Model saved to: saved_models\heart_disease_model_v1.joblib
Metadata saved to: saved_models\heart_disease_model_v1_metadata.json
File size: 0.94 KB
Model verification successful.


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [5]:
# Save the trained model with compression
joblib.dump(
    best_model,
    model_path,
    compress=3,
    protocol=4
)

print("Model saved successfully.")
print("File path:", model_path)
print("File size:", round(model_path.stat().st_size / 1024, 2), "KB")

Model saved successfully.
File path: saved_models\heart_disease_model_v1.joblib
File size: 0.94 KB


In [7]:
# Calculate final test metrics
test_pred = best_model.predict(X_test)

test_precision, test_recall, test_f1 = calculate_metrics(
    y_test, test_pred
)

print("Test Precision:", round(test_precision, 4))
print("Test Recall:", round(test_recall, 4))
print("Test F1 Score:", round(test_f1, 4))

metadata = {
    "model_name": best_model_name,
    "model_version": model_version,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "features": features,
    "target": target,
    "validation_metrics": {
        "precision": float(results_df.iloc[0]["Precision"]),
        "recall": float(results_df.iloc[0]["Recall"]),
        "f1_score": float(results_df.iloc[0]["F1 Score"]),
        "custom_score": float(results_df.iloc[0]["Custom Score"])
    },
    "test_metrics": {
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1_score": float(test_f1)
    },
    "library_versions": {
        "python": os.sys.version.split()[0],
        "scikit_learn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "joblib": joblib.__version__
    }
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print("Metadata saved:", metadata_path)

Test Precision: 0.7692
Test Recall: 0.8333
Test F1 Score: 0.8
Metadata saved: saved_models\heart_disease_model_v1_metadata.json


In [8]:
def calculate_checksum(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

model_checksum = calculate_checksum(model_path)

metadata["sha256"] = model_checksum

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print("SHA-256:", model_checksum)

SHA-256: f494b3057e23240729586e27e436641729b003ab554098fef73e57efabcdc687


In [9]:
# Check file integrity before loading
with open(metadata_path, "r", encoding="utf-8") as file:
    saved_metadata = json.load(file)

current_checksum = calculate_checksum(model_path)

if current_checksum != saved_metadata["sha256"]:
    raise ValueError("Model checksum mismatch. File may have changed.")

# Load the saved model
loaded_model = joblib.load(model_path)

# Compare predictions with the original trained model
original_predictions = best_model.predict(X_test)
loaded_predictions = loaded_model.predict(X_test)

if np.array_equal(original_predictions, loaded_predictions):
    print("Verification successful: predictions match.")
else:
    print("Warning: predictions do not match.")

print("Loaded model:", saved_metadata["model_name"])
print("Version:", saved_metadata["model_version"])
print("Features:", saved_metadata["features"])

Verification successful: predictions match.
Loaded model: Logistic Regression
Version: v1
Features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']


In [10]:
print(json.dumps(saved_metadata, indent=4))

{
    "model_name": "Logistic Regression",
    "model_version": "v1",
    "created_at_utc": "2026-10-21T11:27:05.419958+00:00",
    "features": [
        "age",
        "trestbps",
        "chol",
        "thalach",
        "oldpeak"
    ],
    "target": "target",
    "validation_metrics": {
        "precision": 0.9166666666666666,
        "recall": 0.9166666666666666,
        "f1_score": 0.9166666666666666,
        "custom_score": 0.9166666666666667
    },
    "test_metrics": {
        "precision": 0.7692307692307693,
        "recall": 0.8333333333333334,
        "f1_score": 0.8
    },
    "library_versions": {
        "python": "3.13.14",
        "scikit_learn": "1.9.0",
        "numpy": "2.5.1",
        "pandas": "3.0.3",
        "joblib": "1.5.3"
    },
    "sha256": "f494b3057e23240729586e27e436641729b003ab554098fef73e57efabcdc687"
}


In [11]:
# Example: predict using one test sample
sample = pd.DataFrame(
    [X_test[0]],
    columns=features
)

prediction = loaded_model.predict(sample)[0]

if hasattr(loaded_model, "predict_proba"):
    probability = loaded_model.predict_proba(sample)[0, 1]
    print("Predicted probability of class 1:", round(float(probability), 4))

print("Predicted class:", int(prediction))

Predicted probability of class 1: 0.1353
Predicted class: 0


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
